# Module 05 — Exercise: SAR Coherence and Change Detection

**DIGIHAZ PhD Course** | Disaster Risk Reduction


## Student Information

- **Name:** Yue Qimin
- **Student ID:** Not provided
- **Date:** 2026-10-01


## 1. Why SAR is useful after disasters

SAR is an active microwave system. It can observe day or night and can operate through cloud and smoke, which is important immediately after earthquakes, floods, landslides and storms. Damage may alter backscatter amplitude, coherence and spatial texture.

There is no single universal damage signature: building collapse can reduce double-bounce returns, rubble may increase diffuse scattering, and strong disturbance often reduces coherence.


## 2. Required SAR preprocessing

A reliable pre/post comparison should:
1. apply precise orbit information;
2. remove thermal noise and radiometrically calibrate;
3. reduce speckle while preserving edges;
4. accurately co-register pre/post acquisitions;
5. terrain-correct using a DEM; and
6. mask unsuitable areas such as water, vegetation, layover or shadow when needed.

Pre/post acquisitions should preferably have the same orbit direction and similar incidence angle.


## 3. Synthetic amplitude-change and coherence-loss experiment


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)
h, w = 100, 100

pre_db = rng.normal(-8.0, 1.2, (h,w))
post_db = pre_db + rng.normal(0, 0.5, (h,w))

damage = np.zeros((h,w), dtype=bool)
damage[30:72, 42:78] = True

# Strong structural disturbance
post_db[damage] += rng.normal(-4.5, 1.0, damage.sum())

coh_pre = np.clip(rng.normal(0.78, 0.06, (h,w)), 0, 1)
coh_post = np.clip(coh_pre + rng.normal(0, 0.04, (h,w)), 0, 1)
coh_post[damage] = np.clip(rng.normal(0.28, 0.08, damage.sum()), 0, 1)

amplitude_change = post_db - pre_db
coherence_drop = coh_pre - coh_post

pred = (amplitude_change < -2.5) | (coherence_drop > 0.30)

pd.DataFrame({
    "Indicator":["Mean amplitude change in damage area (dB)",
                 "Mean coherence drop in damage area",
                 "Detected pixels"],
    "Value":[round(amplitude_change[damage].mean(),2),
             round(coherence_drop[damage].mean(),2),
             int(pred.sum())]
})


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(10,8))
axes[0,0].imshow(pre_db, cmap="gray"); axes[0,0].set_title("Pre-event backscatter (dB)")
axes[0,1].imshow(amplitude_change, cmap="RdBu", vmin=-6, vmax=6); axes[0,1].set_title("Amplitude change")
axes[1,0].imshow(coherence_drop, cmap="magma", vmin=0, vmax=0.7); axes[1,0].set_title("Coherence drop")
axes[1,1].imshow(pred, cmap="gray"); axes[1,1].set_title("Damage proxy")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Interpretation

Amplitude change is fast to compute but can be affected by geometry and moisture. Coherence loss is a useful disturbance indicator when a suitable interferometric pair is available and co-registration is accurate. Combining the two can improve robustness, but a radar change is not automatically structural damage.

Operational products should also include buildings, roads, shelters, hospitals and critical infrastructure; acquisition dates, sensor and processing method; and explicit uncertainty / no-data classes.


## 5. Limitations and best practice

Important limitations are speckle, layover, radar shadow, incidence-angle effects, vegetation decorrelation, moisture change and limited ground truth. Best practice is to combine amplitude and coherence, mask unsuitable areas, integrate building footprints and field/optical/UAV evidence, and update the map when better information becomes available.
